In [0]:
import dlt

@dlt.table(
    name="gold.gold_dim_properties",
    comment="Grain: One row per unique property.",
    schema="""
        property_id BIGINT COMMENT 'Unique identifier for the property (Primary Key)',
        property_type STRING COMMENT 'Category of the property',
        destination_id BIGINT COMMENT 'Foreign key to the destination'
    """
)
def gold_dim_properties():
    return dlt.read("silver.silver_properties").where("__is_current = true").select("property_id", "property_type", "destination_id")


@dlt.table(
    name="gold.gold_dim_users",
    comment="Grain: One row per unique user.",
    schema="""
        user_id BIGINT COMMENT 'Unique identifier for the user (Primary Key)',
        country STRING COMMENT 'Country of residence for the user'
    """
)
def gold_dim_users():
    return dlt.read("silver.silver_users").select("user_id", "country")


@dlt.table(
    name="gold.gold_dim_destinations",
    comment="Grain: One row per unique destination.",
    schema="""
        destination_id BIGINT COMMENT 'Unique identifier for the destination (Primary Key)',
        destination STRING COMMENT 'Name of the destination',
        country STRING COMMENT 'Country of the destination'
    """
)
def gold_dim_destinations():
    return dlt.read("silver.silver_destinations").select("destination_id", "destination", "country")


@dlt.table(
    name="gold.gold_fact_bookings",
    comment="Grain: One row per booking transaction.",
    schema="""
        booking_id BIGINT COMMENT 'Unique identifier for the booking (Primary Key)',
        property_id BIGINT COMMENT 'Foreign key to property dimension',
        user_id BIGINT COMMENT 'Foreign key to user dimension',
        destination_id BIGINT COMMENT 'Foreign key to destination dimension (Row-filter dimension)',
        amount DECIMAL(10,2) COMMENT 'Total payment amount for the booking' MASK meridian_t12.gov.mask_amount,
        check_in DATE COMMENT 'Check-in date',
        check_out DATE COMMENT 'Check-out date'
    """,
    row_filter="ROW FILTER meridian_t12.gov.row_filter_by_destination_country ON (destination_id)"
)
def gold_fact_bookings():
    bookings = dlt.read("silver.silver_bookings").alias("b")
    payments = dlt.read("silver.silver_payments").alias("p")
    properties = dlt.read("silver.silver_properties").where("__is_current = true").alias("prop")
    
    joined_df = bookings.join(payments, "booking_id", "left") \
                        .join(properties, "property_id", "left")
    
    return joined_df.select(
        "b.booking_id", 
        "b.property_id", 
        "b.user_id", 
        "prop.destination_id", 
        "p.amount", 
        "b.check_in", 
        "b.check_out"
    )